# 대화 메시지 상태 업데이트 & 실행 방식

이 노트북에서 다루는 것:
1. 메시지 타입 (`HumanMessage` / `AIMessage` / `AnyMessage`)
2. 메시지 상태를 **직접** 누적 업데이트
3. `add_messages` 리듀서로 **자동** 누적 업데이트
4. 그래프 실행 결과를 받는 방법: `invoke` / `ainvoke` / `stream` / `astream`

> 여전히 LLM 호출은 없다. 메시지 객체만 다룬다.

## 메시지 타입

- `HumanMessage` : 사용자(사람)의 메시지
- `AIMessage` : AI(LLM)의 메시지
- `AnyMessage` : `HumanMessage`, `AIMessage` 등을 포함하는 메시지 타입

## 1. 대화 메시지 상태 업데이트하기 (리듀서 없이)

리듀서를 지정하지 않으면 노드 반환값이 **덮어쓰기**다. 그래서 기존 메시지에 직접 이어붙여 반환해야 한다.

- `set_entry_point("node")` : 그래프의 시작 노드를 지정 (= `add_edge(START, "node")`)

In [ ]:
from langchain_core.messages import AnyMessage, AIMessage
from typing_extensions import TypedDict

class State(TypedDict):
    messages: list[AnyMessage]
    extra_field: int

def node(state: State):
    messages = state["messages"]
    new_message = AIMessage("안녕하세요! 무엇을 도와드릴까요?")
    # 기존 messages 에 새 메시지를 직접 이어붙여 반환
    return {"messages": messages + [new_message], "extra_field": 10}

In [ ]:
from langgraph.graph import StateGraph

graph_builder = StateGraph(State)
graph_builder.add_node("node", node)
graph_builder.set_entry_point("node")   # START -> "node"
graph = graph_builder.compile()

그래프 구조를 mermaid PNG 이미지로 시각화한다. (외부 렌더링 API 호출 — 오프라인이면 텍스트로 fallback)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("PNG 렌더 실패:", e)
    print(graph.get_graph().draw_mermaid())

In [ ]:
from langchain_core.messages import HumanMessage

result = graph.invoke({"messages": [HumanMessage("안녕")], "extra_field": 0})
result

In [ ]:
result["messages"]

## 2. 대화 메시지 상태 누적 업데이트하기 (`add_messages`)

`add_messages` 리듀서를 쓰면 노드는 **새 메시지만** 반환하면 된다. 합치기는 리듀서가 처리한다.

또한 `add_messages` 는 dict 형태 메시지(`{"role": ..., "content": ...}`)도 적절한 메시지 객체로 변환해준다.

In [ ]:
from typing_extensions import Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    extra_field: int

def node(state: State):
    new_message = AIMessage("안녕하세요! 무엇을 도와드릴까요?")
    return {"messages": new_message, "extra_field": 10}  # 단일 메시지여도 OK

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("node", node)
graph_builder.set_entry_point("node")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(graph.get_graph().draw_mermaid())

In [ ]:
# dict 형태 메시지 입력 → add_messages 가 자동 변환
input_message = {"role": "user", "content": "안녕하세요."}
result = graph.invoke({"messages": [input_message], "extra_field": 0})

for message in result["messages"]:
    message.pretty_print()   # 보기 좋은 형태로 출력

## 3. 그래프 실행 결과를 받는 방법

| 메서드 | 설명 |
|---|---|
| `invoke` | 동기 단발. 결과가 나올 때까지 멈춤 |
| `ainvoke` | 비동기 단발. 여러 요청을 동시에 처리 |
| `stream` | 중간 결과를 실시간으로 받음 |
| `astream` | 비동기 스트리밍 |

### `invoke` — 하나의 요청 결과를 받을 때까지 멈춤

In [ ]:
graph.invoke({"messages": [input_message], "extra_field": 0})

### `ainvoke` — 비동기로 요청 처리

(Jupyter 셀은 이미 이벤트 루프 안이라 `await` 를 바로 쓸 수 있다.)

In [ ]:
await graph.ainvoke({"messages": [input_message], "extra_field": 0})

### `stream` — 중간 결과를 실시간으로 반환

`stream_mode` 로 출력 형식을 고른다.
- `"values"` : 각 단계의 **현재 상태 값(전체 스냅샷)**
- `"updates"` (기본) : 각 단계의 **상태 업데이트(변경분)만**
- `"messages"` : 메시지와 메타데이터

**`stream_mode="values"`** — 각 단계의 현재 상태 값 출력

In [ ]:
for chunk in graph.stream({"messages": [input_message], "extra_field": 0}, stream_mode="values"):
    print(chunk)
    for state_key, state_value in chunk.items():
        if state_key == "messages":
            state_value[-1].pretty_print()

**`stream_mode="updates"`** (기본) — 각 단계의 상태 업데이트만 출력

In [ ]:
for chunk in graph.stream({"messages": [input_message], "extra_field": 0}, stream_mode="updates"):
    print(chunk)
    for node_name, value in chunk.items():
        print("node:", node_name)

**`stream_mode="messages"`** — 메시지와 메타데이터 출력

In [ ]:
for chunk_msg, metadata in graph.stream({"messages": [input_message], "extra_field": 0}, stream_mode="messages"):
    print(chunk_msg)
    print("content:", chunk_msg.content)
    print("node:", metadata["langgraph_node"])

### `astream` — 비동기 스트리밍

In [ ]:
async for chunk_msg, metadata in graph.astream({"messages": [input_message], "extra_field": 0}, stream_mode="messages"):
    print(chunk_msg.content, "| node:", metadata["langgraph_node"])

## 정리

- 메시지 누적은 `add_messages` 리듀서가 가장 깔끔. 수동 누적은 메커니즘 이해용.
- `set_entry_point("node")` = `add_edge(START, "node")`
- 실행: `invoke`/`ainvoke`(단발), `stream`/`astream`(중간 결과) — `stream_mode` 로 출력 형식 선택

다음: 노드와 엣지를 연결하는 여러 패턴(순차/병렬/조건부)을 다룬다.